In [0]:
%sql
-- ============================================================================
-- Tabela......: mart_risco_credito
-- Camada......: Gold (Data Product)
--
-- Objetivo
-- ----------------------------------------------------------------------------
-- Consolidar mensalmente os principais indicadores da carteira por faixa
-- de score de crédito para suportar análises de risco no Power BI.
--
-- Perguntas de negócio respondidas
-- ----------------------------------------------------------------------------
-- • Qual faixa de risco concentra mais contratos?
-- • Qual faixa concentra maior volume financeiro?
-- • Qual score apresenta maior inadimplência?
-- • Qual score possui pior índice de recebimento?
-- • Como o comportamento do risco evolui ao longo do tempo?
--
-- Grão
-- ----------------------------------------------------------------------------
-- 1 linha por competência (dt_competencia) e faixa de score.
--
-- Fontes
-- ----------------------------------------------------------------------------
-- gold.fact_carteira_credito
-- gold.fact_parcela_credito
-- gold.dim_score
-- ============================================================================

CREATE OR REPLACE TABLE dev_credit_risk.gold.mart_risco_credito

USING DELTA

AS

-- ============================================================================
-- Indicadores da Carteira
-- ============================================================================

WITH carteira AS (

SELECT

    date_trunc('MONTH', dt_contratacao)            AS dt_competencia,

    id_score,

    COUNT(*)                                       AS qtd_contratos,

    SUM(valor_contratado)                          AS valor_contratado,

    AVG(valor_contratado)                          AS ticket_medio

FROM dev_credit_risk.gold.fact_carteira_credito

GROUP BY

    date_trunc('MONTH', dt_contratacao),

    id_score

),

-- ============================================================================
-- Indicadores de Cobrança
-- ============================================================================

cobranca AS (

SELECT

    dt_competencia,

    id_score,

    SUM(valor_parcela)                             AS valor_esperado,

    SUM(COALESCE(valor_pago,0))                    AS valor_recebido,

    SUM(valor_em_aberto)                           AS saldo_em_aberto,

    ROUND(

        SUM(COALESCE(valor_pago,0))
        /
        NULLIF(SUM(valor_parcela),0)

    ,4)                                            AS indice_recebimento,

    ROUND(

        SUM(flag_pagamento_atrasado)
        /
        COUNT(*)

    ,4)                                            AS tx_atraso,

    ROUND(

        SUM(flag_pagamento_default_90)
        /
        COUNT(*)

    ,4)                                            AS tx_default,

    COUNT(

        DISTINCT CASE

            WHEN flag_pagamento_default_90 = 1

            THEN id_contrato

        END

    )                                              AS contratos_default,

    AVG(

        CASE

            WHEN flag_pagamento = 1

            THEN dias_atraso

        END

    )                                              AS media_dias_atraso

FROM dev_credit_risk.gold.fact_parcela_credito

GROUP BY

    dt_competencia,

    id_score

)

-- ============================================================================
-- Resultado Final
-- ============================================================================

SELECT

    c.dt_competencia,

    c.id_score,

    ds.faixa_score,

    ds.risco,

    ----------------------------------------------------------------------------
    -- Carteira
    ----------------------------------------------------------------------------

    c.qtd_contratos,

    c.valor_contratado,

    c.ticket_medio,

    ----------------------------------------------------------------------------
    -- Cobrança
    ----------------------------------------------------------------------------

    b.valor_esperado,

    b.valor_recebido,

    b.indice_recebimento,

    b.saldo_em_aberto,

    ----------------------------------------------------------------------------
    -- Qualidade da Carteira
    ----------------------------------------------------------------------------

    b.contratos_default,

    b.tx_atraso,

    b.tx_default,

    b.media_dias_atraso,

    ----------------------------------------------------------------------------
    -- Auditoria
    ----------------------------------------------------------------------------

    current_timestamp() AS dt_carga

FROM carteira c

LEFT JOIN cobranca b

ON  c.dt_competencia = b.dt_competencia
AND c.id_score       = b.id_score

INNER JOIN dev_credit_risk.gold.dim_score ds

ON c.id_score = ds.id_score

ORDER BY

    c.dt_competencia,

    c.id_score;

In [0]:
%sql
select * from dev_credit_risk.gold.mart_risco_credito